# Ablation: Training Data

Compares effect of sample size (10k vs 25k) and similarity threshold
(0.60, 0.65, 0.70) on SigExt model quality.

**Memory strategy**: LLM loaded once → SigExt variants cycled on CPU.

In [ ]:
!pip install -e ../..
from huggingface_hub import login
login()

In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, load_llm, create_summary_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt
from sm_sip.pipelines import run_inference, run_evaluation
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory

In [ ]:
data = get_test_data(lang='it', num_samples=50, skip_samples=25000)

# Load LLM once
_, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', '8bit')
chain = create_summary_chain(pipe, get_summary_prompt('it', 'source_aware'))

results = {}
for cfg in ['10k-60t', '25k-60t', '25k-65t', '25k-70t']:
    sc = SigExtConfig.from_preset('it', cfg)
    sm, st = load_sigext_model(sc.model_id, device='cpu')
    proc = preprocess_dataset(data, sm, st, lang='it')
    unload_sigext_model(sm, st)
    res = run_inference(proc, chain)
    results[cfg] = run_evaluation(res, lang='it')

clear_gpu_memory()
save_results({'ablation': 'training_data', 'results': results}, 'results/ablation_training_data.json')
print('Done!')